# Batch Speculative Decoding

**COMS 4232 Final Project — Nikhil Sharma (ns3942) & Mertay Dayanc (md4437)**

This notebook implements **Batch Speculative Decoding** (Algorithm 1 from our formalization),
which extends the baseline by sampling $N$ independent draft paths and selecting the best one.

**References:**
- [Yang et al. (2024) — Multi-Candidate Speculative Decoding](https://arxiv.org/abs/2401.06706)
- [Leviathan et al. (2023) — Fast Inference from Transformers via Speculative Decoding](https://arxiv.org/abs/2211.17192)
- Dayanc & Sharma — *Batch Speculative Decoding: Formalization and Unbiasedness*

**Setup:**
- **Draft model ($M_q$):** GPT-2 small (124M)
- **Target model ($M_p$):** GPT-2 XL (1.5B)
- **Batch size $N$:** Number of independent draft paths per round

**Algorithm overview (per round):**

1. **Phase 1 — Drafting:** Sample $N$ independent paths of length $K$ from $M_q$.
2. **Phase 2 — Parallel Evaluation:** Run $M_p$ on all paths to get target distributions $p_i(x)$.
3. **Phase 3 — Path Selection:** For each path $j$, find its rejection index $i_j$ (first position where $r > p_i(\tilde{x}_i) / q_i(\tilde{x}_i)$). Select $j^* = \arg\max_j(i_j)$.
4. **Phase 4 — Verification & Correction:**
   - If $i_{\max} < K$: sample from the corrected residual $p'_{i_{\max}}(x) = \frac{p_{i_{\max}}(x) - A_{\text{eff}}(x)}{1 - \beta_{\text{batch}}}$ where $A_{\text{eff}}(x) = \min\!\big(p_{i_{\max}}(x),\; 1 - (1 - \min(p_{i_{\max}}(x), q_{i_{\max}}(x)))^N\big)$
   - If $i_{\max} = K$: all tokens of path $j^*$ accepted; sample a bonus token from $p_{t+K}(x)$.

## 1. Imports & Model Loading

In [1]:
import torch
import torch.nn.functional as F
from transformers import AutoModelForCausalLM, AutoTokenizer
import time

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

tokenizer = AutoTokenizer.from_pretrained("gpt2")

# Draft model (GPT-2 small, 124M)
draft_model = AutoModelForCausalLM.from_pretrained("gpt2").to(device)
draft_model.eval()

# Target model (GPT-2 XL, 1.5B)
target_model = AutoModelForCausalLM.from_pretrained("gpt2-xl").to(device)
target_model.eval()

print(f"Draft model parameters:  {sum(p.numel() for p in draft_model.parameters()):,}")
print(f"Target model parameters: {sum(p.numel() for p in target_model.parameters()):,}")
print(f"Vocabulary size: {tokenizer.vocab_size}")

Using device: cuda


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

model.safetensors:   0%|          | 0.00/6.43G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/580 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2-xl
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...47}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Draft model parameters:  124,439,808
Target model parameters: 1,557,611,200
Vocabulary size: 50257


## 2. Batch Draft Function (Phase 1)

Sample $N$ independent draft paths, each of length $K$, from the draft model $M_q$.
Each path is generated autoregressively, just like in the baseline — but we do it $N$ times independently.

Returns:
- `all_draft_tokens`: list of $N$ token sequences (each length $K$)
- `all_draft_probs`: list of $N$ probability distribution sequences (each length $K$)

In [2]:
@torch.no_grad()
def batch_draft(
    input_ids: torch.Tensor, N: int, K: int
) -> tuple[list[list[int]], list[list[torch.Tensor]]]:
    """
    Phase 1: Sample N independent draft paths of length K.

    Args:
        input_ids: Prefix token IDs, shape (1, seq_len).
        N: Number of independent draft paths (batch size).
        K: Draft length per path.

    Returns:
        all_draft_tokens: List of N lists, each containing K token IDs.
        all_draft_probs:  List of N lists, each containing K probability
                          distributions (shape (vocab_size,)).
    """
    all_draft_tokens = []
    all_draft_probs = []

    for _ in range(N):
        draft_tokens = []
        draft_probs = []
        current_ids = input_ids.clone()

        for _ in range(K):
            outputs = draft_model(current_ids)
            logits = outputs.logits[0, -1, :]
            probs = F.softmax(logits, dim=-1)
            token = torch.multinomial(probs, num_samples=1).item()

            draft_tokens.append(token)
            draft_probs.append(probs)

            current_ids = torch.cat(
                [current_ids, torch.tensor([[token]], device=device)], dim=1
            )

        all_draft_tokens.append(draft_tokens)
        all_draft_probs.append(draft_probs)

    return all_draft_tokens, all_draft_probs

## 3. Batch Verification (Phases 2–4)

This is the core of the batch algorithm, implementing Phases 2–4:

**Phase 2 — Parallel Evaluation:** Run target model on each path (prefix + draft tokens) to get $p_i(x)$.

**Phase 3 — Path Selection:** For each path $j$, simulate the baseline accept/reject to find the rejection index $i_j$ (the first position where $r > p_i(\tilde{x}_i)/q_i(\tilde{x}_i)$). Select $j^* = \arg\max_j(i_j)$ and $i_{\max} = \max_j(i_j)$.

**Phase 4 — Verification & Correction:**
- If $i_{\max} < K$ (best path was rejected at some position):
  - Compute the **effective acceptance** (Eq. 1 from our proof):
    $$A_{\text{eff}}(x) = \min\!\big(p_{i_{\max}}(x),\; 1 - (1 - \min(p_{i_{\max}}(x),\, q_{i_{\max}}(x)))^N\big)$$
  - Compute $\beta_{\text{batch}} = \sum_x A_{\text{eff}}(x)$
  - Sample from the **corrected residual** (Eq. 3):
    $$p'_{i_{\max}}(x) = \frac{p_{i_{\max}}(x) - A_{\text{eff}}(x)}{1 - \beta_{\text{batch}}}$$
- If $i_{\max} = K$ (full path $j^*$ accepted): sample a bonus token from $p_{t+K}(x)$.

The clipping in $A_{\text{eff}}$ is what prevents the over-sampling bias described in Section 3 of our proof.

In [3]:
@torch.no_grad()
def batch_verify(
    prefix_ids: torch.Tensor,
    all_draft_tokens: list[list[int]],
    all_draft_probs: list[list[torch.Tensor]],
) -> tuple[list[int], int]:
    """
    Phases 2-4: Evaluate all N paths on the target model, select the best
    path, and apply the A_eff correction on rejection.

    Args:
        prefix_ids:       Prefix token IDs, shape (1, n).
        all_draft_tokens: N paths, each a list of K token IDs.
        all_draft_probs:  N paths, each a list of K draft distributions.

    Returns:
        accepted_tokens: Tokens to append (accepted prefix + resampled/bonus).
        n_accepted:      Number of draft tokens accepted from the best path.
    """
    N = len(all_draft_tokens)
    K = len(all_draft_tokens[0])
    n = prefix_ids.shape[1]

    # ------------------------------------------------------------------
    # Phase 2: Run target model on each path to get target distributions.
    # We run N separate forward passes (one per path).
    # ------------------------------------------------------------------
    all_target_probs = []  # N lists, each K+1 distributions (positions n-1 .. n+K-1)
    for j in range(N):
        draft_tensor = torch.tensor([all_draft_tokens[j]], device=device)
        full_ids = torch.cat([prefix_ids, draft_tensor], dim=1)  # (1, n+K)
        outputs = target_model(full_ids)
        # Extract target distributions at positions n-1 through n+K-1
        # logits[0, n-1+t, :] predicts token at position n+t
        path_target_probs = []
        for t in range(K + 1):  # K draft positions + 1 for bonus
            p_t = F.softmax(outputs.logits[0, n - 1 + t, :], dim=-1)
            path_target_probs.append(p_t)
        all_target_probs.append(path_target_probs)

    # ------------------------------------------------------------------
    # Phase 3: Path Selection — find rejection index for each path.
    # For path j, i_j is the first t where r > p_t(x_t) / q_t(x_t).
    # If no rejection, i_j = K (all accepted).
    # ------------------------------------------------------------------
    rejection_indices = []  # i_j for each path
    for j in range(N):
        i_j = K  # assume all accepted unless we find a rejection
        for t in range(K):
            x_t = all_draft_tokens[j][t]
            p_x = all_target_probs[j][t][x_t].item()
            q_x = all_draft_probs[j][t][x_t].item()

            if q_x == 0:
                acceptance_prob = 0.0
            else:
                acceptance_prob = min(1.0, p_x / q_x)

            r = torch.rand(1).item()
            if r > acceptance_prob:
                i_j = t
                break
        rejection_indices.append(i_j)

    # Select best path: j* = argmax(i_j), i_max = max(i_j)
    i_max = max(rejection_indices)
    j_star = rejection_indices.index(i_max)

    # ------------------------------------------------------------------
    # Phase 4: Verification & Correction
    # ------------------------------------------------------------------
    # Accepted prefix: tokens 0..i_max-1 from path j*
    accepted_tokens = list(all_draft_tokens[j_star][:i_max])
    n_accepted = i_max

    if i_max < K:
        # Best path was rejected at position i_max.
        # Apply A_eff correction (Eq. 1 from our proof).
        p_imax = all_target_probs[j_star][i_max]  # target dist at rejection position
        q_imax = all_draft_probs[j_star][i_max]    # draft dist at rejection position

        # A_eff(x) = min(p(x), 1 - (1 - min(p(x), q(x)))^N)
        overlap = torch.min(p_imax, q_imax)
        raw_batch_accept = 1.0 - (1.0 - overlap).pow(N)
        A_eff = torch.min(p_imax, raw_batch_accept)

        # beta_batch = sum_x A_eff(x)
        beta_batch = A_eff.sum().item()

        # Corrected residual: p'(x) = (p(x) - A_eff(x)) / (1 - beta_batch)
        residual = p_imax - A_eff
        residual = torch.clamp(residual, min=0.0)  # numerical safety
        residual_sum = residual.sum()

        if residual_sum > 0:
            residual = residual / residual_sum
        else:
            # Fallback (shouldn't happen given the proof)
            residual = p_imax

        resampled_token = torch.multinomial(residual, num_samples=1).item()
        accepted_tokens.append(resampled_token)
    else:
        # All K tokens of path j* accepted — sample bonus token from p_{t+K}
        bonus_probs = all_target_probs[j_star][K]  # target dist at position n+K
        bonus_token = torch.multinomial(bonus_probs, num_samples=1).item()
        accepted_tokens.append(bonus_token)

    return accepted_tokens, n_accepted

## 4. Batch Speculative Decoding — Outer Loop

Repeatedly calls `batch_draft()` and `batch_verify()` until we reach `max_new_tokens` or EOS.
Tracks acceptance metrics to compare against the baseline.

In [4]:
@torch.no_grad()
def batch_speculative_decode(
    prompt: str,
    N: int = 2,
    K: int = 4,
    max_new_tokens: int = 50,
) -> dict:
    """
    Generate text using batch speculative decoding.

    Args:
        prompt:         Input text prompt.
        N:              Number of independent draft paths per round.
        K:              Draft length per path.
        max_new_tokens: Maximum new tokens to generate.

    Returns:
        Dictionary with generation results and metrics.
    """
    input_ids = tokenizer.encode(prompt, return_tensors="pt").to(device)
    generated_ids = input_ids.clone()
    eos_token_id = tokenizer.eos_token_id

    total_accepted = 0
    total_drafted = 0
    rounds = 0
    target_calls = 0
    draft_calls = 0

    start_time = time.time()

    tokens_generated = 0
    while tokens_generated < max_new_tokens:
        # Phase 1: Draft N paths of length K
        all_draft_tokens, all_draft_probs = batch_draft(generated_ids, N, K)
        draft_calls += N * K  # N paths, K forward passes each

        # Phases 2-4: Verify and correct
        accepted_tokens, n_accepted = batch_verify(
            generated_ids, all_draft_tokens, all_draft_probs
        )
        target_calls += N  # one target forward pass per path

        # Bookkeeping
        total_accepted += n_accepted
        total_drafted += K  # K tokens were proposed from the best path
        rounds += 1

        # Append accepted tokens
        new_tokens = torch.tensor([accepted_tokens], device=device)
        generated_ids = torch.cat([generated_ids, new_tokens], dim=1)
        tokens_generated += len(accepted_tokens)

        # Check for EOS
        if eos_token_id in accepted_tokens:
            break

    elapsed = time.time() - start_time
    output_text = tokenizer.decode(generated_ids[0], skip_special_tokens=True)
    acceptance_rate = total_accepted / total_drafted if total_drafted > 0 else 0.0

    return {
        "text": output_text,
        "tokens_generated": tokens_generated,
        "rounds": rounds,
        "total_accepted": total_accepted,
        "total_drafted": total_drafted,
        "acceptance_rate": acceptance_rate,
        "time": elapsed,
        "target_calls": target_calls,
        "draft_calls": draft_calls,
        "N": N,
        "K": K,
    }

## 5. Vanilla Autoregressive Baseline

Same as in the baseline notebook — target model only, one forward pass per token.

In [5]:
@torch.no_grad()
def autoregressive_generate(
    prompt: str,
    max_new_tokens: int = 50,
) -> dict:
    """Standard autoregressive generation using the target model."""
    input_ids = tokenizer.encode(prompt, return_tensors="pt").to(device)
    generated_ids = input_ids.clone()
    eos_token_id = tokenizer.eos_token_id

    start_time = time.time()

    tokens_generated = 0
    for _ in range(max_new_tokens):
        outputs = target_model(generated_ids)
        logits = outputs.logits[0, -1, :]
        probs = F.softmax(logits, dim=-1)
        token = torch.multinomial(probs, num_samples=1).item()

        generated_ids = torch.cat(
            [generated_ids, torch.tensor([[token]], device=device)], dim=1
        )
        tokens_generated += 1

        if token == eos_token_id:
            break

    elapsed = time.time() - start_time
    output_text = tokenizer.decode(generated_ids[0], skip_special_tokens=True)

    return {
        "text": output_text,
        "tokens_generated": tokens_generated,
        "time": elapsed,
        "target_calls": tokens_generated,
    }

## 6. Single-Run Comparison

Compare batch speculative decoding (N=2, K=4) against vanilla autoregressive on a single prompt.
The key question: does sampling multiple draft paths improve the acceptance rate compared to the baseline (N=1)?

In [6]:
prompt = "The future of artificial intelligence is"
N = 2
K = 4
max_new_tokens = 50

print(f"Prompt: \"{prompt}\"")
print(f"Batch size N = {N}, Draft length K = {K}, max_new_tokens = {max_new_tokens}")
print("=" * 70)

# --- Batch Speculative Decoding ---
print("\n[Batch Speculative Decoding]")
batch_result = batch_speculative_decode(prompt, N=N, K=K, max_new_tokens=max_new_tokens)

print(f"Generated text:\n  {batch_result['text']}\n")
print(f"  Tokens generated:      {batch_result['tokens_generated']}")
print(f"  Rounds:                {batch_result['rounds']}")
print(f"  Draft tokens proposed: {batch_result['total_drafted']}")
print(f"  Draft tokens accepted: {batch_result['total_accepted']}")
print(f"  Acceptance rate:       {batch_result['acceptance_rate']:.2%}")
print(f"  Target model calls:    {batch_result['target_calls']} ({N} per round)")
print(f"  Draft model calls:     {batch_result['draft_calls']} ({N}x{K} per round)")
print(f"  Wall-clock time:       {batch_result['time']:.2f}s")

# --- Vanilla Autoregressive ---
print("\n" + "=" * 70)
print("\n[Vanilla Autoregressive (target model only)]")
ar_result = autoregressive_generate(prompt, max_new_tokens=max_new_tokens)

print(f"Generated text:\n  {ar_result['text']}\n")
print(f"  Tokens generated:   {ar_result['tokens_generated']}")
print(f"  Target model calls: {ar_result['target_calls']}")
print(f"  Wall-clock time:    {ar_result['time']:.2f}s")

# --- Summary ---
print("\n" + "=" * 70)
print("\n[Summary]")
print(f"  Batch SD used {batch_result['target_calls']} target calls "
      f"vs {ar_result['target_calls']} for autoregressive.")
if ar_result['time'] > 0:
    speedup = ar_result['time'] / batch_result['time']
    print(f"  Wall-clock speedup: {speedup:.2f}x")
print(f"  Token acceptance rate: {batch_result['acceptance_rate']:.2%}")

Prompt: "The future of artificial intelligence is"
Batch size N = 2, Draft length K = 4, max_new_tokens = 50

[Batch Speculative Decoding]
Generated text:
  The future of artificial intelligence is riding on the SSL patch. Patching vulnerable software is an all-encompassing management job that requires corruption detection, security bug tracking and security-related training. If patches are skipped, the odds of hackers finding vulnerabilities are high.

At the workshop

  Tokens generated:      52
  Rounds:                15
  Draft tokens proposed: 60
  Draft tokens accepted: 37
  Acceptance rate:       61.67%
  Target model calls:    30 (2 per round)
  Draft model calls:     120 (2x4 per round)
  Wall-clock time:       3.76s


[Vanilla Autoregressive (target model only)]
Generated text:
  The future of artificial intelligence is really up to us," Hanks said on Monday's edition of "Turning Points." The actor added, "Bots and other technology are working serious amounts of schedules, s

## 7. Averaged Comparison (Multiple Runs)

Run both methods multiple times and average for stable metrics.
This lets us reliably compare batch SD acceptance rates against the baseline notebook's ~47% average.

In [7]:
prompt = "The future of artificial intelligence is"
N = 2
K = 4
max_new_tokens = 50
n_runs = 10

print(f"Running {n_runs} trials | N = {N}, K = {K}, max_new_tokens = {max_new_tokens}")
print(f"Prompt: \"{prompt}\"")
print("=" * 70)

# Collect metrics
batch_times = []
batch_acceptance_rates = []
batch_target_calls = []
batch_draft_calls = []
batch_tokens = []

ar_times = []
ar_target_calls = []

for i in range(n_runs):
    br = batch_speculative_decode(prompt, N=N, K=K, max_new_tokens=max_new_tokens)
    batch_times.append(br["time"])
    batch_acceptance_rates.append(br["acceptance_rate"])
    batch_target_calls.append(br["target_calls"])
    batch_draft_calls.append(br["draft_calls"])
    batch_tokens.append(br["tokens_generated"])

    ar = autoregressive_generate(prompt, max_new_tokens=max_new_tokens)
    ar_times.append(ar["time"])
    ar_target_calls.append(ar["target_calls"])

    print(f"  Trial {i+1:2d}: batch {br['time']:.2f}s (accept {br['acceptance_rate']:.0%}, "
          f"rounds {br['rounds']}) | AR {ar['time']:.2f}s")

# Averages
avg_batch_time = sum(batch_times) / n_runs
avg_ar_time = sum(ar_times) / n_runs
avg_acceptance = sum(batch_acceptance_rates) / n_runs
avg_batch_target = sum(batch_target_calls) / n_runs
avg_batch_draft = sum(batch_draft_calls) / n_runs
avg_ar_target = sum(ar_target_calls) / n_runs
avg_batch_tokens = sum(batch_tokens) / n_runs

print("\n" + "=" * 70)
print(f"\n[Averaged over {n_runs} runs]")
print(f"  Batch Speculative Decoding (N={N}, K={K}):")
print(f"    Avg time:            {avg_batch_time:.2f}s")
print(f"    Avg acceptance rate: {avg_acceptance:.2%}")
print(f"    Avg target calls:    {avg_batch_target:.1f}")
print(f"    Avg draft calls:     {avg_batch_draft:.1f}")
print(f"    Avg tokens generated:{avg_batch_tokens:.1f}")
print(f"  Autoregressive:")
print(f"    Avg time:            {avg_ar_time:.2f}s")
print(f"    Avg target calls:    {avg_ar_target:.1f}")
print(f"\n  Avg wall-clock speedup: {avg_ar_time / avg_batch_time:.2f}x")
print(f"  Avg target call reduction: {avg_ar_target:.0f} -> {avg_batch_target:.0f}")
print(f"\n  Compare to baseline SD (N=1): ~47% acceptance rate")
print(f"  Batch SD (N={N}): {avg_acceptance:.2%} acceptance rate")

Running 10 trials | N = 2, K = 4, max_new_tokens = 50
Prompt: "The future of artificial intelligence is"
  Trial  1: batch 2.76s (accept 64%, rounds 14) | AR 2.34s
  Trial  2: batch 2.27s (accept 73%, rounds 13) | AR 2.39s
  Trial  3: batch 2.68s (accept 68%, rounds 14) | AR 2.57s
  Trial  4: batch 3.50s (accept 40%, rounds 20) | AR 2.39s
  Trial  5: batch 2.69s (accept 58%, rounds 15) | AR 2.69s
  Trial  6: batch 2.31s (accept 71%, rounds 13) | AR 2.42s
  Trial  7: batch 3.05s (accept 49%, rounds 17) | AR 2.46s
  Trial  8: batch 2.79s (accept 64%, rounds 14) | AR 2.45s
  Trial  9: batch 2.88s (accept 53%, rounds 16) | AR 2.47s
  Trial 10: batch 3.19s (accept 62%, rounds 15) | AR 2.47s


[Averaged over 10 runs]
  Batch Speculative Decoding (N=2, K=4):
    Avg time:            2.81s
    Avg acceptance rate: 60.23%
    Avg target calls:    30.2
    Avg draft calls:     120.8
    Avg tokens generated:50.7
  Autoregressive:
    Avg time:            2.46s
    Avg target calls:    50.0

  Av